# Mat & Comp – ML 10: Bosques aleatorios y ensambles
## La sabiduría de la multitud

**Curso:** Machine learning (Sección 3) · **Duración sugerida:** 2 sesiones de 50 minutos · **Requisito:** ML 9

### Objetivos de aprendizaje
Al terminar esta lección podrás:
1. Explicar con simulaciones por qué el **promedio** o el **voto** de muchos modelos imperfectos puede ser mejor que cada uno.
2. Crear muestras **bootstrap** y entrenar varios árboles con ellas (**bagging**), a mano con numpy.
3. Entender qué agrega un **bosque aleatorio**: elegir al azar las características en cada pregunta, para que los árboles sean distintos.
4. Usar `RandomForestClassifier`, su **error fuera de la bolsa** y la **importancia** de las variables.
5. Reconocer cuándo un bosque **no** es la mejor opción y conocer otro ensamble: el **boosting**.

## 🔍 ¿Cuánto pesa el buey?

En 1906, en una feria de ganado en Inglaterra, 787 personas intentaron adivinar el peso de un buey. Casi nadie acertó. Pero el científico Francis Galton calculó el **promedio** de todas las respuestas y quedó sorprendido: se equivocaba por menos de un 1 %.

✏️ *¿Quién crees que adivina mejor: la persona **típica** de la feria o el **promedio** de todas las personas? ¿Por qué?*

**Mi respuesta:**

Esta idea, la **sabiduría de la multitud**, es la base de algunos de los modelos más usados del machine learning: los **ensambles**. En ML 5 un solo árbol de decisión sobreajustaba con facilidad. Hoy vamos a sembrar **un bosque entero**. 🌲🌲🌲

## 1. La multitud, simulada

Simulemos la feria: el buey pesa 540 kg, y cada persona responde el peso real más un error al azar (unos se pasan, otros se quedan cortos).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
real = 540
respuestas = real + rng.normal(0, 60, 787)        # cada persona se equivoca unos 60 kg

errores_personas = np.abs(respuestas - real)
print(f"Error de la persona típica (mediana): {np.median(errores_personas):.1f} kg")
print(f"Error del promedio de todos:          {abs(respuestas.mean() - real):.1f} kg")
print(f"Personas que lo hicieron mejor que el promedio: {np.sum(errores_personas < abs(respuestas.mean() - real))} de 787")

plt.figure(figsize=(7, 3))
plt.hist(respuestas, bins=30, color="lightgray", edgecolor="gray")
plt.axvline(real, color="green", lw=2, label="peso real")
plt.axvline(respuestas.mean(), color="purple", lw=2, ls="--", label="promedio de la multitud")
plt.xlabel("peso adivinado (kg)"); plt.legend()
plt.show()

Los errores de unos y otros **se cancelan** al promediar. Pero hay una condición importante: los errores deben ser **independientes**. Si todos copian al vecino más ruidoso, la multitud se equivoca en bloque:

In [ ]:
sesgo_comun = 50                                  # todos fueron influidos por un rumor: "pesa como 600"
copiones = real + sesgo_comun + rng.normal(0, 20, 787)
print(f"Error del promedio de la multitud que copia: {abs(copiones.mean() - real):.1f} kg")

### Votar en lugar de promediar

Para **clasificar** no promediamos: **votamos**. Imagina que cada votante acierta un 60 % de las veces, de forma independiente. ¿Con qué frecuencia acierta la **mayoría**?

In [ ]:
def acierto_mayoria(n_votantes, p, repeticiones=20000):
    aciertos = rng.random((repeticiones, n_votantes)) < p     # True = este votante acertó
    return np.mean(aciertos.sum(axis=1) > n_votantes / 2)

votantes = [1, 3, 5, 11, 21, 51, 101, 201]
resultados = [acierto_mayoria(n, 0.6) for n in votantes]
for n, r in zip(votantes, resultados):
    print(f"{n:4} votantes de 60 %: la mayoría acierta el {r:.1%}")

plt.figure(figsize=(6, 3))
plt.plot(votantes, resultados, "o-")
plt.xscale("log")
plt.xlabel("número de votantes"); plt.ylabel("acierto de la mayoría")
plt.grid(alpha=0.3)
plt.show()

Con 201 votantes que aciertan apenas un poco más que tirar una moneda, la mayoría acierta casi siempre. (Esto se conoce como el **teorema del jurado de Condorcet**, de 1785.)

Entonces la receta es: **muchos modelos, distintos entre sí, que se equivoquen de forma independiente**. ¿Cómo conseguimos muchos árboles distintos si solo tenemos un conjunto de datos?

## 2. Un árbol es inestable

Usaremos las dos **medias lunas** de `make_moons` (como en el reto de ML 8). Entrenemos árboles **sin límite de profundidad** con cuatro muestras distintas de los datos y comparemos sus fronteras:

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(400, noise=0.3, random_state=0)
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.5, random_state=0)

g1, g2 = np.meshgrid(np.linspace(-2, 3, 250), np.linspace(-1.5, 2, 250))
malla = np.c_[g1.ravel(), g2.ravel()]

def dibujar(eje, prediccion, titulo):
    eje.contourf(g1, g2, prediccion.reshape(g1.shape), levels=20, cmap="coolwarm_r", alpha=0.5)
    eje.scatter(X_ent[:, 0], X_ent[:, 1], c=y_ent, cmap="coolwarm_r", s=10, edgecolor="k", linewidth=0.3)
    eje.set_title(titulo, fontsize=10)
    eje.set_xticks([]); eje.set_yticks([])

fig, ejes = plt.subplots(1, 4, figsize=(15, 3.2))
for k, eje in enumerate(ejes):
    idx = rng.choice(len(X_ent), size=len(X_ent), replace=True)   # una muestra al azar (ya verás por qué así)
    arbol = DecisionTreeClassifier(random_state=0).fit(X_ent[idx], y_ent[idx])
    dibujar(eje, arbol.predict(malla), f"árbol {k + 1}: {arbol.score(X_pru, y_pru):.0%} en prueba")
plt.show()

Cada árbol memoriza su muestra y dibuja islas y bordes en escalera muy distintos: pequeños cambios en los datos, árboles muy diferentes. Eso es **sobreajuste** (ML 5 y ML 7)… pero también es justo lo que necesitamos: ¡modelos **distintos** que se equivocan en lugares distintos!

## 3. Bagging, a mano

### Muestras bootstrap

Una **muestra bootstrap** se crea sacando datos **con reposición**: como sacar papelitos de una bolsa y devolverlos antes de sacar el siguiente. Tiene el mismo tamaño que los datos originales, pero algunos datos salen repetidos y otros no salen.

In [ ]:
bolsa = np.arange(10)
for _ in range(3):
    muestra = rng.choice(bolsa, size=10, replace=True)
    print(np.sort(muestra), "→", len(np.unique(muestra)), "distintos")

En promedio, una muestra bootstrap contiene cerca del **63 %** de los datos originales; el otro 37 % queda «fuera de la bolsa» (volveremos a esto).

### Bagging = bootstrap + votar

**Bagging** (de *bootstrap aggregating*) es:
1. Crear muchas muestras bootstrap.
2. Entrenar un árbol con cada una.
3. Para predecir, **votar**.

In [ ]:
def bagging(X, y, n_arboles, semilla=0):
    r = np.random.default_rng(semilla)
    arboles = []
    for _ in range(n_arboles):
        idx = r.choice(len(X), size=len(X), replace=True)
        arboles.append(DecisionTreeClassifier(random_state=0).fit(X[idx], y[idx]))
    return arboles

def votar(arboles, X):
    votos = np.array([a.predict(X) for a in arboles])    # una fila por árbol
    return (votos.mean(axis=0) > 0.5).astype(int)        # gana la mayoría

arboles = bagging(X_ent, y_ent, 200)
for n in [1, 5, 25, 200]:
    exactitud = np.mean(votar(arboles[:n], X_pru) == y_pru)
    print(f"{n:3} árboles: {exactitud:.1%} en prueba")

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(13, 3.6))
for eje, n in zip(ejes, [1, 25, 200]):
    proporcion = np.mean([a.predict(malla) for a in arboles[:n]], axis=0)   # fracción de árboles que dicen 1
    dibujar(eje, proporcion, f"{n} árbol(es): {np.mean(votar(arboles[:n], X_pru) == y_pru):.0%} en prueba")
plt.show()

Con muchos árboles, las islas raras de cada uno se cancelan en la votación. La frontera se vuelve más **suave**, y en lugar de un «sí o no» tenemos una **proporción de votos**, que funciona como una probabilidad (ML 8).

En estas medias lunas la exactitud sube poco: hay tanto ruido (puntos de un color metidos en la zona del otro) que ningún modelo pasa mucho del 90 %. La gran diferencia aparecerá con datos más complejos.

## 4. Bosques aleatorios: árboles todavía más distintos

En el bagging, todos los árboles ven las mismas características. Si hay una característica muy fuerte, casi todos empiezan con la **misma pregunta** y terminan pareciéndose: como la multitud que copia del ejemplo del buey.

El **bosque aleatorio** (Leo Breiman, 2001) agrega un truco: en **cada pregunta** de cada árbol, solo se permite elegir entre unas pocas características **al azar**. Así los árboles se ven obligados a ser diferentes.

Para verlo necesitamos datos con muchas características: los dígitos de 8 × 8 píxeles de ML 4 (64 características).

In [ ]:
from sklearn.datasets import load_digits
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier

dig = load_digits()
Xd_ent, Xd_pru, yd_ent, yd_pru = train_test_split(dig.data, dig.target, test_size=0.3, random_state=0)

modelos = {
    "un árbol": DecisionTreeClassifier(random_state=0),
    "bagging (200 árboles)": BaggingClassifier(DecisionTreeClassifier(), n_estimators=200, random_state=0),
    "bosque aleatorio (200 árboles)": RandomForestClassifier(n_estimators=200, random_state=0),
}
for nombre, m in modelos.items():
    m.fit(Xd_ent, yd_ent)
    print(f"{nombre:32}: {m.score(Xd_pru, yd_pru):.1%} en prueba")

Del árbol solo al bosque hay un salto enorme. Por defecto, `RandomForestClassifier` deja elegir entre $\sqrt{64} = 8$ píxeles al azar en cada pregunta (el parámetro `max_features`).

¿Qué tan parecidos son los árboles entre sí? Midamos con qué frecuencia dos árboles dan la **misma respuesta** en los datos de prueba:

In [ ]:
def acuerdo_promedio(ensamble, X):
    preds = np.array([a.predict(X) for a in ensamble.estimators_[:30]])
    pares = [np.mean(preds[i] == preds[j]) for i in range(30) for j in range(i + 1, 30)]
    return np.mean(pares)

print(f"Bagging:          los árboles coinciden el {acuerdo_promedio(modelos['bagging (200 árboles)'], Xd_pru):.1%} de las veces")
print(f"Bosque aleatorio: los árboles coinciden el {acuerdo_promedio(modelos['bosque aleatorio (200 árboles)'], Xd_pru):.1%} de las veces")

Los árboles del bosque se parecen **menos** entre sí: se equivocan de forma más independiente, y por eso la votación funciona mejor.

✏️ *¿Por qué cada árbol del bosque, por separado, probablemente es **peor** que un árbol de bagging? ¿Por qué el bosque completo es mejor?*

## 5. Validación gratis: el error fuera de la bolsa

Cada árbol no vio cerca del 37 % de los datos de entrenamiento (los que quedaron fuera de su muestra bootstrap). Podemos evaluar cada dato usando **solo** los árboles que no lo vieron. Es como una validación cruzada (ML 7) que viene incluida:

In [ ]:
bosque = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0)
bosque.fit(Xd_ent, yd_ent)
print(f"Exactitud fuera de la bolsa (solo con entrenamiento): {bosque.oob_score_:.1%}")
print(f"Exactitud en la prueba:                              {bosque.score(Xd_pru, yd_pru):.1%}")

## 6. ¿Qué variables usa el bosque?

Como en ML 5, el bosque puede decir qué tan útil fue cada característica (promediando sobre todos sus árboles). Con los dígitos, cada característica es un píxel, ¡así que podemos dibujar la importancia como una imagen!

In [ ]:
plt.figure(figsize=(3.5, 3.5))
plt.imshow(bosque.feature_importances_.reshape(8, 8), cmap="viridis")
plt.colorbar(label="importancia")
plt.title("Píxeles más útiles")
plt.axis("off")
plt.show()

Las columnas de los bordes izquierdo y derecho casi no importan (casi siempre están en blanco); los dígitos se distinguen en el centro de la imagen.

Ahora con los estudiantes de ML 1, para predecir si aprobaron:

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/emersonjleon/compumath/main/lecciones/seccion3_ml/datos/estudiantes.csv"
datos = pd.read_csv(URL)
datos["horas_sueno"] = datos["horas_sueno"].fillna(datos["horas_sueno"].mean())
datos["deporte"] = (datos["deporte"] == "sí").astype(int)
columnas = ["horas_estudio", "horas_sueno", "horas_redes", "grado", "deporte"]
Xe = datos[columnas].to_numpy()
ye = (datos["nota_matematicas"] >= 3.0).astype(int).to_numpy()

bosque_e = RandomForestClassifier(n_estimators=300, random_state=0).fit(Xe, ye)
orden = np.argsort(bosque_e.feature_importances_)
plt.figure(figsize=(6, 2.8))
plt.barh(np.array(columnas)[orden], bosque_e.feature_importances_[orden], color="seagreen")
plt.xlabel("importancia")
plt.title("¿Qué usa el bosque para decidir si aprobó?")
plt.show()

## 7. ¿Siempre gana el bosque?

No. Comparemos con validación cruzada, en los mismos estudiantes, varios modelos que ya conoces:

In [ ]:
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

grupos = KFold(5, shuffle=True, random_state=0)
print("Clasificar ¿aprobó? (exactitud):")
for nombre, m in [("árbol (profundidad 3)", DecisionTreeClassifier(max_depth=3, random_state=0)),
                  ("bosque aleatorio", RandomForestClassifier(n_estimators=300, random_state=0)),
                  ("regresión logística", make_pipeline(StandardScaler(), LogisticRegression()))]:
    print(f"  {nombre:22}: {cross_val_score(m, Xe, ye, cv=grupos).mean():.1%}")

nota = datos["nota_matematicas"].to_numpy()
print("\nPredecir la nota (R²):")
for nombre, m in [("bosque aleatorio", RandomForestRegressor(n_estimators=300, random_state=0)),
                  ("regresión lineal", LinearRegression())]:
    print(f"  {nombre:22}: {cross_val_score(m, Xe, nota, cv=grupos).mean():.3f}")

En estos datos, la relación con la nota es casi una **recta** (ML 2 y ML 9). Un modelo lineal la captura perfectamente con pocos parámetros, mientras que el bosque la aproxima con escalones. Los bosques brillan cuando las relaciones son **complicadas** (como los dígitos o las medias lunas) y hay suficientes datos.

**No hay un modelo que siempre gane.** Por eso comparamos con validación cruzada.

## 8. Otro ensamble: boosting

En el bosque, los árboles crecen **a la vez** y de forma independiente. En el **boosting**, crecen **uno tras otro**: cada árbol nuevo, pequeño, se concentra en corregir los errores de los anteriores. Es como una fila de estudiantes donde cada uno revisa y corrige el trabajo del anterior.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

refuerzo = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=0)
refuerzo.fit(X_ent, y_ent)
bosque_lunas = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_ent, y_ent)

fig, ejes = plt.subplots(1, 2, figsize=(10, 3.6))
dibujar(ejes[0], bosque_lunas.predict_proba(malla)[:, 1], f"bosque aleatorio: {bosque_lunas.score(X_pru, y_pru):.0%}")
dibujar(ejes[1], refuerzo.predict_proba(malla)[:, 1], f"boosting: {refuerzo.score(X_pru, y_pru):.0%}")
plt.show()

Bosques aleatorios y boosting (con variantes como XGBoost o LightGBM) son los modelos que más ganan en competencias con datos en forma de **tabla**. Para imágenes, sonido y texto, en cambio, ganan las **redes neuronales** de la Sección 4.

## 9. Errores comunes

**Error 1: pedir importancias antes de entrenar.**

In [ ]:
RandomForestClassifier().feature_importances_

Primero `fit`, después preguntar.

**Error 2: olvidar `random_state`.** El bosque usa el azar (las muestras y las características). Sin `random_state`, cada ejecución da resultados un poco distintos.

**Error 3: creer que más árboles causan sobreajuste.** Más árboles **no** hacen que el bosque memorice más: la exactitud se estabiliza. Lo que sí cuesta es **tiempo**.

**Error 4: extrapolar con un bosque de regresión.** Cada árbol predice promedios de los datos que vio, así que el bosque **nunca** predice fuera del rango de los datos de entrenamiento (lo verás en el ejercicio 5).

**Error 5: tomar la importancia como causa.** Que el bosque use mucho una característica no significa que esa característica **cause** el resultado (ML 1).

## 🤔 ¿Qué pasa si…?

Usa la celda de abajo. Antes de ejecutar, **predice** el resultado.

1. …en `acierto_mayoria` usas votantes que aciertan el **40 %**? ¿Ayuda tener más votantes?
2. …en `bagging` cambias `replace=True` por `replace=False`? ¿Siguen siendo distintos los árboles? ¿Qué exactitud da la votación?
3. …entrenas el bosque de los dígitos con `max_features=64` (todas las características)? ¿En qué se convierte?
4. …usas `max_depth=2` en los árboles del bosque de dígitos? ¿Pueden 200 árboles muy simples compensarlo?
5. …entrenas `RandomForestClassifier(n_estimators=1)` en los dígitos? ¿Es igual que un árbol solo?

In [ ]:
# Prueba aquí tus ideas

## ✍️ Ejercicios

**Ejercicio 1 (fácil). ¿Cuántos quedan dentro?** Repite 10 000 veces: saca una muestra bootstrap de tamaño 10 de los números 0 a 9 y cuenta cuántos números **distintos** salen. ¿Cuál es el promedio? Compáralo con la fórmula $10 \cdot \big(1 - (9/10)^{10}\big)$.

**Ejercicio 2 (fácil). El jurado de cinco.** Cinco jueces independientes aciertan cada uno el 70 % de las veces. Calcula con `acierto_mayoria` la probabilidad de que la mayoría acierte. ¿Y si fueran 15 jueces?

**Ejercicio 3 (medio). ¿Cuántos árboles?** Entrena bosques con 1, 2, 5, 10, 20, 50, 100, 200 y 500 árboles en los dígitos y grafica la exactitud en prueba (eje x logarítmico). ¿A partir de cuántos árboles deja de mejorar? ¿Empeora en algún momento?

**Ejercicio 4 (medio). `max_features`.** Con 200 árboles en los dígitos, prueba `max_features` = 1, 2, 4, 8, 16, 32 y 64. Grafica la exactitud en prueba. ¿Qué pasa en los extremos?

**Ejercicio 5 (avanzado). Bosques que no extrapolan.** Genera los datos de frenado de ML 9 (`rng = np.random.default_rng(1)`, `v = np.sort(rng.uniform(20, 110, 30)).round(0)`, `d = (0.28 * v + v ** 2 / 178 + rng.normal(0, 5, 30)).round(1)`). Entrena un `RandomForestRegressor` y una parábola (ML 9) solo con los carros que iban a **menos de 80 km/h**, y predice la distancia a 100 y 120 km/h. Grafica ambas curvas entre 0 y 130 km/h. ¿Qué le pasa al bosque?

## 🏆 Reto opcional: tu propio bosque

Construye un bosque aleatorio a mano: usa la función `bagging` de la sección 3, pero crea cada árbol con `DecisionTreeClassifier(max_features="sqrt", random_state=...)` (cada árbol con una semilla distinta), y en `votar` usa el voto más frecuente entre 10 clases (pista: `np.bincount` en cada columna, o `scipy.stats.mode`). Entrénalo en los dígitos con 200 árboles y compara su exactitud con `RandomForestClassifier`.

## 🎬 Cierre

Hoy aprendiste que:
- El **promedio** o el **voto** de muchos modelos puede superar a cada uno, siempre que se equivoquen de forma **independiente**.
- Un árbol profundo es **inestable**: con datos un poco distintos, cambia mucho.
- **Bagging**: muestras **bootstrap** (con reposición) + un árbol por muestra + votación.
- El **bosque aleatorio** además elige al azar las características en cada pregunta, para que los árboles sean más distintos.
- Los datos **fuera de la bolsa** dan una validación gratis, y el bosque mide la **importancia** de las variables.
- El **boosting** encadena árboles que corrigen errores. Y ningún modelo gana siempre.

**Para pensar:** los dígitos tienen 64 características, pero los bordes casi no aportan, y muchos píxeles vecinos dicen casi lo mismo. ¿Podríamos describir cada imagen con **muchos menos números** sin perder lo importante? En la próxima lección veremos la **reducción de dimensionalidad con PCA**, y podremos dibujar los 1797 dígitos en una sola gráfica de dos dimensiones. 🗺️